In [2]:
# Check that Fabric can see the Files folder
print(notebookutils.fs.ls("Files"))

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://9daf80d2-893f-4b03-9c54-7f4e5818ac9a@onelake.dfs.fabric.microsoft.com/b3b3a228-8c6d-4b1d-b3e6-b87e9aed4d29/Files/items.csv, name=items.csv, size=101841), FileInfo(path=abfss://9daf80d2-893f-4b03-9c54-7f4e5818ac9a@onelake.dfs.fabric.microsoft.com/b3b3a228-8c6d-4b1d-b3e6-b87e9aed4d29/Files/stores.csv, name=stores.csv, size=1387), FileInfo(path=abfss://9daf80d2-893f-4b03-9c54-7f4e5818ac9a@onelake.dfs.fabric.microsoft.com/b3b3a228-8c6d-4b1d-b3e6-b87e9aed4d29/Files/train.csv, name=train.csv, size=4997452288), FileInfo(path=abfss://9daf80d2-893f-4b03-9c54-7f4e5818ac9a@onelake.dfs.fabric.microsoft.com/b3b3a228-8c6d-4b1d-b3e6-b87e9aed4d29/Files/transactions.csv, name=transactions.csv, size=1552637)]


In [3]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/train.csv")
)

df.show(5, truncate=False)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 5, Finished, Available, Finished, False)

+---+----------+---------+--------+----------+-----------+
|id |date      |store_nbr|item_nbr|unit_sales|onpromotion|
+---+----------+---------+--------+----------+-----------+
|0  |2013-01-01|25       |103665  |7.0       |NULL       |
|1  |2013-01-01|25       |105574  |1.0       |NULL       |
|2  |2013-01-01|25       |105575  |2.0       |NULL       |
|3  |2013-01-01|25       |108079  |1.0       |NULL       |
|4  |2013-01-01|25       |108701  |1.0       |NULL       |
+---+----------+---------+--------+----------+-----------+
only showing top 5 rows



In [4]:
print("Rows:", df.count())
print("Columns:", df.columns)

df.printSchema()

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 6, Finished, Available, Finished, False)

Rows: 125497040
Columns: ['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion']
root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- item_nbr: integer (nullable = true)
 |-- unit_sales: double (nullable = true)
 |-- onpromotion: boolean (nullable = true)



In [5]:
from pyspark.sql import functions as F

print("Null values:")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 7, Finished, Available, Finished, False)

Null values:
+---+----+---------+--------+----------+-----------+
| id|date|store_nbr|item_nbr|unit_sales|onpromotion|
+---+----+---------+--------+----------+-----------+
|  0|   0|        0|       0|         0|   21657651|
+---+----+---------+--------+----------+-----------+



In [6]:
df_clean = df.withColumn(
    "onpromotion",
    F.coalesce(F.col("onpromotion"), F.lit(False))
)

print("Null values after cleaning:")

df_clean.select(
    F.count(F.when(F.col("onpromotion").isNull(), 1)).alias("onpromotion_nulls")
).show()

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 8, Finished, Available, Finished, False)

Null values after cleaning:
+-----------------+
|onpromotion_nulls|
+-----------------+
|                0|
+-----------------+



In [7]:
daily_sales = (
    df_clean
    .groupBy("date")
    .agg(
        F.sum("unit_sales").alias("total_sales"),
        F.countDistinct("store_nbr").alias("active_stores"),
        F.countDistinct("item_nbr").alias("active_items")
    )
    .orderBy("date")
)

daily_sales.show(10)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 10, Finished, Available, Finished, False)

+----------+------------------+-------------+------------+
|      date|       total_sales|active_stores|active_items|
+----------+------------------+-------------+------------+
|2013-01-01|2511.6190000000006|            1|         578|
|2013-01-02|496092.41799999896|           46|        1583|
|2013-01-03| 361429.2309999994|           46|        1586|
|2013-01-04|354459.67700000026|           46|        1586|
|2013-01-05| 477350.1209999992|           46|        1595|
|2013-01-06|519695.40099999943|           46|        1565|
|2013-01-07| 336122.8010000001|           46|        1589|
|2013-01-08|318347.77799999993|           46|        1588|
|2013-01-09|302530.80899999954|           46|        1589|
|2013-01-10|258982.00300000006|           46|        1590|
+----------+------------------+-------------+------------+
only showing top 10 rows



In [8]:
# Load supporting Favorita datasets

items = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/items.csv")
)

stores = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/stores.csv")
)

transactions = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/transactions.csv")
)

oil = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/oil.csv")
)

holidays = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/holidays_events.csv")
)

print("Items:")
items.show(5)

print("Stores:")
stores.show(5)

print("Transactions:")
transactions.show(5)

print("Oil:")
oil.show(5)

print("Holidays:")
holidays.show(5)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 13, Finished, Available, Finished, False)

Items:
+--------+------------+-----+----------+
|item_nbr|      family|class|perishable|
+--------+------------+-----+----------+
|   96995|   GROCERY I| 1093|         0|
|   99197|   GROCERY I| 1067|         0|
|  103501|    CLEANING| 3008|         0|
|  103520|   GROCERY I| 1028|         0|
|  103665|BREAD/BAKERY| 2712|         1|
+--------+------------+-----+----------+
only showing top 5 rows

Stores:
+---------+-------------+--------------------+----+-------+
|store_nbr|         city|               state|type|cluster|
+---------+-------------+--------------------+----+-------+
|        1|        Quito|           Pichincha|   D|     13|
|        2|        Quito|           Pichincha|   D|     13|
|        3|        Quito|           Pichincha|   D|      8|
|        4|        Quito|           Pichincha|   D|      9|
|        5|Santo Domingo|Santo Domingo de ...|   D|      4|
+---------+-------------+--------------------+----+-------+
only showing top 5 rows

Transactions:
+----------+

In [9]:
# Prepare train data for monthly forecasting

train_monthly = (
    df_clean
    .withColumn("month", F.date_trunc("month", F.col("date")))
    .join(
        items.select("item_nbr", "family"),
        on="item_nbr",
        how="left"
    )
    .groupBy("store_nbr", "family", "month")
    .agg(
        F.sum("unit_sales").alias("sales"),
        F.sum(
            F.when(F.col("onpromotion") == True, 1).otherwise(0)
        ).alias("promo")
    )
)

train_monthly.show(10)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 16, Finished, Available, Finished, False)

+---------+----------------+-------------------+------------------+-----+
|store_nbr|          family|              month|             sales|promo|
+---------+----------------+-------------------+------------------+-----+
|        6| LAWN AND GARDEN|2013-01-01 00:00:00|             135.0|    0|
|       11|LIQUOR,WINE,BEER|2013-01-01 00:00:00|             863.0|    0|
|       32|          BEAUTY|2013-01-01 00:00:00|              12.0|    0|
|       23|          BEAUTY|2013-01-01 00:00:00|              18.0|    0|
|       28|            DELI|2013-02-01 00:00:00|          4949.331|    0|
|       30|    FROZEN FOODS|2013-02-01 00:00:00|            207.37|    0|
|       38|    BREAD/BAKERY|2013-02-01 00:00:00|         12183.414|    0|
|       40|   PERSONAL CARE|2013-02-01 00:00:00|            4014.0|    0|
|       45|            DELI|2013-02-01 00:00:00|14571.437999999995|    0|
|       14|       GROCERY I|2013-03-01 00:00:00|           78465.0|    0|
+---------+----------------+----------

In [10]:
# Create complete store × family × month grid
# This ensures missing months are represented as zero sales.

months = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2013-01-01")),
                F.to_date(F.lit("2017-07-01")),
                F.expr("interval 1 month")
            )
        ).alias("month")
    )
)

stores_list = train_monthly.select("store_nbr").distinct()
families_list = train_monthly.select("family").distinct()

calendar_grid = (
    stores_list
    .crossJoin(families_list)
    .crossJoin(months)
)

monthly_data = (
    calendar_grid
    .join(
        train_monthly,
        on=["store_nbr", "family", "month"],
        how="left"
    )
    .fillna({
        "sales": 0.0,
        "promo": 0.0
    })
    .orderBy("store_nbr", "family", "month")
)

print("Monthly modeling dataset:")
monthly_data.show(10)

print("Number of rows:", monthly_data.count())

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 19, Finished, Available, Finished, False)

Monthly modeling dataset:
+---------+----------+----------+-----+-----+
|store_nbr|    family|     month|sales|promo|
+---------+----------+----------+-----+-----+
|        1|AUTOMOTIVE|2013-01-01| 68.0|    0|
|        1|AUTOMOTIVE|2013-02-01| 38.0|    0|
|        1|AUTOMOTIVE|2013-03-01| 59.0|    0|
|        1|AUTOMOTIVE|2013-04-01| 66.0|    0|
|        1|AUTOMOTIVE|2013-05-01| 79.0|    0|
|        1|AUTOMOTIVE|2013-06-01| 66.0|    0|
|        1|AUTOMOTIVE|2013-07-01| 70.0|    0|
|        1|AUTOMOTIVE|2013-08-01| 98.0|    0|
|        1|AUTOMOTIVE|2013-09-01| 63.0|    0|
|        1|AUTOMOTIVE|2013-10-01| 54.0|    0|
+---------+----------+----------+-----+-----+
only showing top 10 rows

Number of rows: 98010


In [11]:
# Add store information
monthly_data = (
    monthly_data
    .join(
        stores.select(
            "store_nbr",
            "city",
            "state",
            "type",
            "cluster"
        ),
        on="store_nbr",
        how="left"
    )
)

# Prepare monthly transactions
transactions_monthly = (
    transactions
    .withColumn("month", F.date_trunc("month", F.col("date")))
    .groupBy("store_nbr", "month")
    .agg(
        F.sum("transactions").alias("transactions")
    )
)

# Prepare monthly oil price
oil_monthly = (
    oil
    .withColumn("month", F.date_trunc("month", F.col("date")))
    .groupBy("month")
    .agg(
        F.avg("dcoilwtico").alias("dcoilwtico")
    )
)

# Join transactions and oil
monthly_data = (
    monthly_data
    .join(
        transactions_monthly,
        on=["store_nbr", "month"],
        how="left"
    )
    .join(
        oil_monthly,
        on="month",
        how="left"
    )
)

print("After store, transaction and oil features:")
monthly_data.show(10)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 22, Finished, Available, Finished, False)

After store, transaction and oil features:
+----------+---------+--------------+------+-----+--------+--------+----+-------+------------+------------------+
|     month|store_nbr|        family| sales|promo|    city|   state|type|cluster|transactions|        dcoilwtico|
+----------+---------+--------------+------+-----+--------+--------+----+-------+------------+------------------+
|2013-03-01|       31|PREPARED FOODS|1395.0|    0|Babahoyo|Los Rios|   B|     10|       39591|           92.9385|
|2013-02-01|       31|PREPARED FOODS|1203.0|    0|Babahoyo|Los Rios|   B|     10|       34401| 95.30894736842104|
|2013-01-01|       31|PREPARED FOODS|1153.0|    0|Babahoyo|Los Rios|   B|     10|       35549| 94.75666666666666|
|2013-09-01|       31|PREPARED FOODS|1653.0|    0|Babahoyo|Los Rios|   B|     10|       36077|          106.2895|
|2013-06-01|       31|PREPARED FOODS|1557.0|    0|Babahoyo|Los Rios|   B|     10|       39219|           95.7725|
|2013-07-01|       31|PREPARED FOODS|1737.0| 

In [12]:
# Prepare holiday features

hol = (
    holidays
    .filter(F.col("transferred") == False)
    .withColumn("month", F.date_trunc("month", F.col("date")))
)

# National holidays per month
national_holidays = (
    hol
    .filter(F.col("locale") == "National")
    .groupBy("month")
    .agg(
        F.count("*").alias("hol_national")
    )
)

# Regional holidays per month and state
regional_holidays = (
    hol
    .filter(F.col("locale") == "Regional")
    .groupBy("month", F.col("locale_name").alias("state"))
    .agg(
        F.count("*").alias("hol_regional")
    )
)

# Local holidays per month and city
local_holidays = (
    hol
    .filter(F.col("locale") == "Local")
    .groupBy("month", F.col("locale_name").alias("city"))
    .agg(
        F.count("*").alias("hol_local")
    )
)

# Add holiday features to monthly dataset
monthly_data = (
    monthly_data
    .join(national_holidays, on="month", how="left")
    .join(regional_holidays, on=["month", "state"], how="left")
    .join(local_holidays, on=["month", "city"], how="left")
    .fillna({
        "hol_national": 0,
        "hol_regional": 0,
        "hol_local": 0
    })
)

print("After adding holiday features:")
monthly_data.show(10)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 25, Finished, Available, Finished, False)

After adding holiday features:
+----------+--------+--------+---------+--------------+------+-----+----+-------+------------+------------------+------------+------------+---------+
|     month|    city|   state|store_nbr|        family| sales|promo|type|cluster|transactions|        dcoilwtico|hol_national|hol_regional|hol_local|
+----------+--------+--------+---------+--------------+------+-----+----+-------+------------+------------------+------------+------------+---------+
|2013-03-01|Babahoyo|Los Rios|       31|PREPARED FOODS|1395.0|    0|   B|     10|       39591|           92.9385|           0|           0|        0|
|2013-02-01|Babahoyo|Los Rios|       31|PREPARED FOODS|1203.0|    0|   B|     10|       34401| 95.30894736842104|           2|           0|        0|
|2013-01-01|Babahoyo|Los Rios|       31|PREPARED FOODS|1153.0|    0|   B|     10|       35549| 94.75666666666666|           3|           0|        0|
|2013-09-01|Babahoyo|Los Rios|       31|PREPARED FOODS|1653.0|    0| 

In [13]:
# PRISM-style leakage-safe feature engineering

from pyspark.sql.window import Window

# Order each store-family series chronologically
w = (
    Window
    .partitionBy("store_nbr", "family")
    .orderBy("month")
)

# Lag features
monthly_features = (
    monthly_data
    .withColumn("lag_1", F.lag("sales", 1).over(w))
    .withColumn("lag_2", F.lag("sales", 2).over(w))
    .withColumn("lag_3", F.lag("sales", 3).over(w))
    .withColumn("lag_6", F.lag("sales", 6).over(w))
    .withColumn("lag_12", F.lag("sales", 12).over(w))
    .withColumn("lag_13", F.lag("sales", 13).over(w))
    
    # Rolling statistics using ONLY previous months
    .withColumn(
        "roll_mean_3",
        (
            F.col("lag_1") +
            F.col("lag_2") +
            F.col("lag_3")
        ) / 3
    )
    .withColumn(
        "roll_mean_6",
        F.avg("sales").over(
            w.rowsBetween(-6, -1)
        )
    )
    .withColumn(
        "roll_std_6",
        F.stddev("sales").over(
            w.rowsBetween(-6, -1)
        )
    )
    
    # Momentum and year-over-year ratios
    .withColumn(
        "mom_ratio",
        (F.col("lag_1") + 1) / (F.col("lag_2") + 1)
    )
    .withColumn(
        "yoy_ratio",
        (F.col("lag_1") + 1) / (F.col("lag_13") + 1)
    )
    
    # Previous-month external features
    .withColumn("promo_lag_1", F.lag("promo", 1).over(w))
    .withColumn("trans_lag_1", F.lag("transactions", 1).over(w))
    .withColumn("oil_lag_1", F.lag("dcoilwtico", 1).over(w))
    
    # Calendar feature
    .withColumn("month_of_year", F.month("month"))
)

print("PRISM features created:")
monthly_features.show(10)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 28, Finished, Available, Finished, False)

PRISM features created:
+----------+-----+---------+---------+---------+-----+-----+----+-------+------------+------------------+------------+------------+---------+-----+-----+-----+-----+------+------+-----------+-----------+----------+---------+---------+-----------+-----------+------------------+-------------+
|     month| city|    state|store_nbr|   family|sales|promo|type|cluster|transactions|        dcoilwtico|hol_national|hol_regional|hol_local|lag_1|lag_2|lag_3|lag_6|lag_12|lag_13|roll_mean_3|roll_mean_6|roll_std_6|mom_ratio|yoy_ratio|promo_lag_1|trans_lag_1|         oil_lag_1|month_of_year|
+----------+-----+---------+---------+---------+-----+-----+----+-------+------------+------------------+------------+------------+---------+-----+-----+-----+-----+------+------+-----------+-----------+----------+---------+---------+-----------+-----------+------------------+-------------+
|2013-01-01|Quito|Pichincha|        1|BABY CARE|  0.0|    0|   D|     13|       47471| 94.7566666666

In [14]:
# Prepare modeling data for LightGBM

model_data = (
    monthly_features
    .filter(F.col("month") >= F.to_date(F.lit("2014-02-01")))
    .dropna(subset=[
        "lag_1", "lag_2", "lag_3",
        "lag_6", "lag_12",
        "roll_mean_3", "roll_mean_6",
        "roll_std_6", "mom_ratio",
        "yoy_ratio", "promo_lag_1",
        "trans_lag_1", "oil_lag_1"
    ])
)

print("Rows ready for modeling:", model_data.count())

# Convert ONLY the small aggregated modeling table
pdf = model_data.toPandas()

print("Pandas shape:", pdf.shape)
pdf.head()

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 31, Finished, Available, Finished, False)

Rows ready for modeling: 70389
Pandas shape: (70389, 29)


,month,city,state,store_nbr,family,sales,promo,type,cluster,transactions,...,lag_13,roll_mean_3,roll_mean_6,roll_std_6,mom_ratio,yoy_ratio,promo_lag_1,trans_lag_1,oil_lag_1,month_of_year
0,2014-02-01,Quito,Pichincha,1,BABY CARE,0.0,0,D,13,43819.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0,47417,94.617143,2
1,2014-03-01,Quito,Pichincha,1,BABY CARE,0.0,0,D,13,44311.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0,43819,100.817368,3
2,2014-04-01,Quito,Pichincha,1,BABY CARE,0.0,0,D,13,46035.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0,44311,100.803810,4
3,2014-05-01,Quito,Pichincha,1,BABY CARE,0.0,0,D,13,47225.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0,46035,102.069048,5
4,2014-06-01,Quito,Pichincha,1,BABY CARE,0.0,0,D,13,45680.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0,47225,102.177143,6


In [17]:
# ============================================================
# PRISM LIGHTGBM SALES FORECASTING
# Rolling-origin backtest + holdout evaluation
# ============================================================

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# ------------------------------------------------------------
# 1. Feature configuration
# ------------------------------------------------------------

FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "roll_mean_3", "roll_mean_6", "roll_std_6",
    "mom_ratio", "yoy_ratio",
    "promo", "promo_lag_1",
    "trans_lag_1", "oil_lag_1",
    "hol_national", "hol_regional", "hol_local",
    "month_of_year",
    "store_nbr", "family", "type", "cluster", "city"
]

CATEGORICAL = [
    "store_nbr", "family", "type", "cluster", "city"
]

# Make categorical columns actual pandas categories
for c in CATEGORICAL:
    pdf[c] = pdf[c].astype("category")

# Target
pdf["y"] = np.log1p(pdf["sales"])

# Make sure data is chronologically ordered
pdf = pdf.sort_values(["month", "store_nbr", "family"]).reset_index(drop=True)

print("Training rows:", len(pdf))
print("Features:", len(FEATURES))

# ------------------------------------------------------------
# 2. LightGBM model
# ------------------------------------------------------------

def create_model():
    return LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.2,
        n_estimators=700,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1
    )

# ------------------------------------------------------------
# 3. Rolling-origin backtesting
# ------------------------------------------------------------

backtest_months = pd.period_range(
    "2016-02", "2017-01", freq="M"
)

holdout_months = pd.period_range(
    "2017-02", "2017-07", freq="M"
)

results = []

for current_month in list(backtest_months) + list(holdout_months):

    train = pdf[pdf["month"] < current_month]
    test = pdf[pdf["month"] == current_month].copy()

    if len(test) == 0:
        continue

    model = create_model()

    model.fit(
        train[FEATURES],
        train["sales"],
        categorical_feature=CATEGORICAL
    )

    test["prediction"] = np.clip(
        model.predict(test[FEATURES]),
        0,
        None
    )

    test["split"] = (
        "backtest"
        if current_month in backtest_months
        else "holdout"
    )

    test["forecast_month"] = str(current_month)

    results.append(
        test[
            [
                "store_nbr",
                "family",
                "month",
                "sales",
                "prediction",
                "lag_1",
                "lag_12",
                "split",
                "forecast_month"
            ]
        ]
    )

    print(
        current_month,
        "| train:", len(train),
        "| test:", len(test)
    )

results_df = pd.concat(results, ignore_index=True)

print("\nForecasting complete.")
print("Result rows:", len(results_df))

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 35, Finished, Available, Finished, False)

Training rows: 70389
Features: 23


2016-02 | train: 38940 | test: 1749


2016-03 | train: 40689 | test: 1749


2016-04 | train: 42438 | test: 1749


...

2017-03 | train: 61545 | test: 1749


2017-04 | train: 63294 | test: 1749


2017-05 | train: 65043 | test: 1782


2017-06 | train: 66825 | test: 1782


2017-07 | train: 68607 | test: 1782

Forecasting complete.
Result rows: 31449


In [16]:
# Fix month format for time-based backtesting
pdf["month"] = pd.to_datetime(pdf["month"]).dt.to_period("M")

print("Month type:", pdf["month"].dtype)
print("First month:", pdf["month"].min())
print("Last month:", pdf["month"].max())

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 34, Finished, Available, Finished, False)

Month type: period[M]
First month: 2014-02
Last month: 2017-07


In [18]:
# ============================================================
# PRISM MODEL EVALUATION
# ============================================================

def calculate_metrics(data):
    actual = data["sales"]
    pred = data["prediction"]

    error = actual - pred

    rmse = np.sqrt(np.mean(error ** 2))

    # WAPE is safer than row-level MAPE when actual sales can be zero
    wape = np.sum(np.abs(error)) / np.sum(np.abs(actual))

    # Total monthly MAPE
    monthly = (
        data.groupby("forecast_month")
        .agg(
            actual=("sales", "sum"),
            predicted=("prediction", "sum")
        )
    )

    monthly["ape"] = (
        np.abs(monthly["actual"] - monthly["predicted"])
        / monthly["actual"]
    )

    mape = monthly["ape"].mean()

    bias = (
        np.sum(pred) - np.sum(actual)
    ) / np.sum(actual)

    return {
        "RMSE": rmse,
        "WAPE": wape,
        "MAPE": mape,
        "Bias": bias
    }


# Separate backtest and holdout
backtest_results = results_df[
    results_df["split"] == "backtest"
]

holdout_results = results_df[
    results_df["split"] == "holdout"
]

backtest_metrics = calculate_metrics(backtest_results)
holdout_metrics = calculate_metrics(holdout_results)

print("========== BACKTEST ==========")
for metric, value in backtest_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\n========== HOLDOUT ==========")
for metric, value in holdout_metrics.items():
    print(f"{metric}: {value:.4f}")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 36, Finished, Available, Finished, False)

========== BACKTEST ==========
RMSE: 5267.6421
WAPE: 0.0983
MAPE: 0.0588
Bias: 0.0211

========== HOLDOUT ==========
RMSE: 4357.4127
WAPE: 0.0774
MAPE: 0.0361
Bias: -0.0124


In [20]:
# ============================================================
# HOLDOUT BASELINE COMPARISON
# ============================================================

# Bring lag_2 and lag_3 into the forecast results
lag_values = pdf[
    ["store_nbr", "family", "month", "lag_1", "lag_2", "lag_3", "lag_12"]
].copy()

comparison = holdout_results.merge(
    lag_values,
    on=["store_nbr", "family", "month"],
    how="left",
    suffixes=("", "_pdf")
)

# Baselines
comparison["naive"] = comparison["lag_1"]
comparison["seasonal_naive"] = comparison["lag_12"]

comparison["moving_average_3"] = (
    comparison["lag_1"] +
    comparison["lag_2"] +
    comparison["lag_3"]
) / 3


def evaluate_forecast(actual, predicted):
    error = actual - predicted

    rmse = np.sqrt(np.mean(error ** 2))

    wape = (
        np.sum(np.abs(error))
        / np.sum(np.abs(actual))
    )

    return rmse, wape


models = {
    "PRISM LightGBM": comparison["prediction"],
    "Naive": comparison["naive"],
    "Seasonal Naive": comparison["seasonal_naive"],
    "Moving Average 3": comparison["moving_average_3"]
}

print("========== HOLDOUT BASELINE COMPARISON ==========\n")

for name, prediction in models.items():

    valid = prediction.notna()

    rmse, wape = evaluate_forecast(
        comparison.loc[valid, "sales"],
        prediction.loc[valid]
    )

    print(name)
    print(f"  RMSE : {rmse:.2f}")
    print(f"  WAPE : {wape:.4%}")
    print()

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 38, Finished, Available, Finished, False)

========== HOLDOUT BASELINE COMPARISON ==========

PRISM LightGBM
  RMSE : 4357.41
  WAPE : 7.7422%

Naive
  RMSE : 5002.73
  WAPE : 9.6106%

Seasonal Naive
  RMSE : 8021.07
  WAPE : 13.1646%

Moving Average 3
  RMSE : 4982.51
  WAPE : 8.8687%



In [21]:
# ============================================================
# CONFORMAL-STYLE PREDICTION INTERVALS
# Calibrate on backtest residuals only
# ============================================================

# Work on a copy
interval_results = results_df.copy()

# Log-scale actual and prediction
interval_results["actual_log"] = np.log1p(
    interval_results["sales"]
)

interval_results["pred_log"] = np.log1p(
    interval_results["prediction"]
)

# Backtest residuals ONLY
backtest_interval = interval_results[
    interval_results["split"] == "backtest"
].copy()

abs_residuals = np.abs(
    backtest_interval["actual_log"]
    - backtest_interval["pred_log"]
)

# 80% and 95% calibration quantiles
q80 = float(np.quantile(abs_residuals, 0.80))
q95 = float(np.quantile(abs_residuals, 0.95))

print("80% residual quantile:", q80)
print("95% residual quantile:", q95)

# Apply intervals to every forecast
interval_results["lower_80"] = np.clip(
    np.expm1(interval_results["pred_log"] - q80),
    0,
    None
)

interval_results["upper_80"] = np.expm1(
    interval_results["pred_log"] + q80
)

interval_results["lower_95"] = np.clip(
    np.expm1(interval_results["pred_log"] - q95),
    0,
    None
)

interval_results["upper_95"] = np.expm1(
    interval_results["pred_log"] + q95
)

# ------------------------------------------------------------
# Evaluate coverage on HOLDOUT only
# ------------------------------------------------------------

holdout_interval = interval_results[
    interval_results["split"] == "holdout"
]

coverage_80 = (
    (
        (holdout_interval["sales"] >= holdout_interval["lower_80"]) &
        (holdout_interval["sales"] <= holdout_interval["upper_80"])
    ).mean()
)

coverage_95 = (
    (
        (holdout_interval["sales"] >= holdout_interval["lower_95"]) &
        (holdout_interval["sales"] <= holdout_interval["upper_95"])
    ).mean()
)

print("\n========== PREDICTION INTERVALS ==========")
print(f"80% interval coverage: {coverage_80:.2%}")
print(f"95% interval coverage: {coverage_95:.2%}")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 39, Finished, Available, Finished, False)

80% residual quantile: 0.34565305612223973
95% residual quantile: 0.9214321805854575

========== PREDICTION INTERVALS ==========
80% interval coverage: 84.62%
95% interval coverage: 96.08%


In [22]:
# ============================================================
# CONFORMAL-STYLE PREDICTION INTERVALS
# ============================================================

interval_results = results_df.copy()

# Log-scale actual and prediction
interval_results["actual_log"] = np.log1p(interval_results["sales"])
interval_results["pred_log"] = np.log1p(interval_results["prediction"])

# Use BACKTEST residuals only for calibration
backtest_interval = interval_results[
    interval_results["split"] == "backtest"
].copy()

abs_residuals = np.abs(
    backtest_interval["actual_log"]
    - backtest_interval["pred_log"]
)

# 80% and 95% interval widths
q80 = float(np.quantile(abs_residuals, 0.80))
q95 = float(np.quantile(abs_residuals, 0.95))

print("80% residual quantile:", q80)
print("95% residual quantile:", q95)

# Create prediction intervals
interval_results["lower_80"] = np.clip(
    np.expm1(interval_results["pred_log"] - q80),
    0,
    None
)

interval_results["upper_80"] = np.expm1(
    interval_results["pred_log"] + q80
)

interval_results["lower_95"] = np.clip(
    np.expm1(interval_results["pred_log"] - q95),
    0,
    None
)

interval_results["upper_95"] = np.expm1(
    interval_results["pred_log"] + q95
)

# Check coverage on HOLDOUT only
holdout_interval = interval_results[
    interval_results["split"] == "holdout"
]

coverage_80 = (
    (
        (holdout_interval["sales"] >= holdout_interval["lower_80"]) &
        (holdout_interval["sales"] <= holdout_interval["upper_80"])
    ).mean()
)

coverage_95 = (
    (
        (holdout_interval["sales"] >= holdout_interval["lower_95"]) &
        (holdout_interval["sales"] <= holdout_interval["upper_95"])
    ).mean()
)

print("\n========== PREDICTION INTERVALS ==========")
print(f"80% interval coverage: {coverage_80:.2%}")
print(f"95% interval coverage: {coverage_95:.2%}")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 40, Finished, Available, Finished, False)

80% residual quantile: 0.34565305612223973
95% residual quantile: 0.9214321805854575

========== PREDICTION INTERVALS ==========
80% interval coverage: 84.62%
95% interval coverage: 96.08%


In [23]:
# ============================================================
# SAVE FORECAST RESULTS FOR POWER BI
# ============================================================

# Monthly total forecast table
powerbi_forecast = (
    interval_results
    .groupby(["forecast_month", "split"], as_index=False)
    .agg(
        actual_sales=("sales", "sum"),
        predicted_sales=("prediction", "sum"),
        lower_80=("lower_80", "sum"),
        upper_80=("upper_80", "sum"),
        lower_95=("lower_95", "sum"),
        upper_95=("upper_95", "sum")
    )
)

# Convert month to date
powerbi_forecast["forecast_month"] = pd.to_datetime(
    powerbi_forecast["forecast_month"]
)

print(powerbi_forecast)

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 41, Finished, Available, Finished, False)

   forecast_month     split  actual_sales  predicted_sales      lower_80  \
0      2016-02-01  backtest  2.194741e+07     2.313584e+07  1.637408e+07   
1      2016-03-01  backtest  2.313178e+07     2.429405e+07  1.719381e+07   
2      2016-04-01  backtest  2.596008e+07     2.472148e+07  1.749633e+07   
3      2016-05-01  backtest  2.476341e+07     2.840373e+07  2.010247e+07   
4      2016-06-01  backtest  2.220922e+07     2.449183e+07  1.733379e+07   
5      2016-07-01  backtest  2.346267e+07     2.233480e+07  1.580713e+07   
6      2016-08-01  backtest  2.245241e+07     2.400442e+07  1.698882e+07   
7      2016-09-01  backtest  2.241745e+07     2.270666e+07  1.607032e+07   
8      2016-10-01  backtest  2.397455e+07     2.363651e+07  1.672844e+07   
9      2016-11-01  backtest  2.464258e+07     2.608335e+07  1.846020e+07   
10     2016-12-01  backtest  2.930393e+07     2.885870e+07  2.042447e+07   
11     2017-01-01  backtest  2.632812e+07     2.405798e+07  1.702672e+07   
12     2017-

In [24]:
# Convert to Spark DataFrame
forecast_spark = spark.createDataFrame(powerbi_forecast)

# Save as a Delta table
(
    forecast_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_sales_forecast")
)

print("Saved table: fact_sales_forecast")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 42, Finished, Available, Finished, False)

Saved table: fact_sales_forecast


In [26]:
# Remove the old table with the conflicting schema
spark.sql("DROP TABLE IF EXISTS fact_sales_forecast")

print("Old table removed.")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 44, Finished, Available, Finished, False)

Old table removed.


In [29]:
# Recreate the Power BI forecast table with a clean schema

powerbi_forecast = (
    interval_results
    .groupby(["forecast_month", "split"], as_index=False)
    .agg(
        actual_sales=("sales", "sum"),
        predicted_sales=("prediction", "sum"),
        lower_80=("lower_80", "sum"),
        upper_80=("upper_80", "sum"),
        lower_95=("lower_95", "sum"),
        upper_95=("upper_95", "sum")
    )
)

# Force forecast_month to a consistent string type
powerbi_forecast["forecast_month"] = (
    powerbi_forecast["forecast_month"].astype(str)
)

forecast_spark = spark.createDataFrame(powerbi_forecast)

(
    forecast_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_sales_forecast")
)

print("SUCCESS: fact_sales_forecast saved to Lakehouse")

StatementMeta(, 1f381bf5-30a8-41e0-b7e0-bc952a1ecb93, 47, Finished, Available, Finished, False)

SUCCESS: fact_sales_forecast saved to Lakehouse


In [2]:
# Check the forecast results already saved in the Lakehouse

check = spark.sql("""
    SELECT
        forecast_month,
        split,
        actual_sales,
        predicted_sales,
        lower_80,
        upper_80,
        lower_95,
        upper_95
    FROM fact_sales_forecast
    ORDER BY forecast_month DESC
""")

display(check.limit(20))

StatementMeta(, 81a1f730-ee43-42bd-ba9c-046c220080b9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, def93f9d-4fa6-491c-baf1-332d5bec4366)

In [3]:
print("Forecast months:")

spark.sql("""
    SELECT
        MIN(forecast_month) AS first_month,
        MAX(forecast_month) AS last_month
    FROM fact_sales_forecast
""").show()

print("Splits:")

spark.sql("""
    SELECT
        split,
        COUNT(*) AS rows
    FROM fact_sales_forecast
    GROUP BY split
""").show()

StatementMeta(, 81a1f730-ee43-42bd-ba9c-046c220080b9, 6, Finished, Available, Finished, False)

Forecast months:
+-----------+----------+
|first_month|last_month|
+-----------+----------+
|    2016-02|   2017-07|
+-----------+----------+

Splits:
+--------+----+
|   split|rows|
+--------+----+
|backtest|  12|
| holdout|   6|
+--------+----+



In [4]:
# ============================================================
# NEXT-MONTH SALES FORECAST
# Train on all available historical data and predict August 2017
# ============================================================

# Make sure the data is sorted
pdf = pdf.sort_values(["month", "store_nbr", "family"]).reset_index(drop=True)

# Latest month available in our modeling data
latest_month = pdf["month"].max()

# Next month to forecast
forecast_month = latest_month + 1

print("Latest available month :", latest_month)
print("Forecast month          :", forecast_month)

# ------------------------------------------------------------
# 1. Train final model on ALL available historical data
# ------------------------------------------------------------

final_model = create_model()

final_model.fit(
    pdf[FEATURES],
    pdf["sales"],
    categorical_feature=CATEGORICAL
)

print("Final model trained successfully.")

# ------------------------------------------------------------
# 2. Create the data for the next month
# ------------------------------------------------------------

future_df = pdf[pdf["month"] == latest_month].copy()

future_df["month"] = forecast_month

# ------------------------------------------------------------
# 3. Predict next-month sales
# ------------------------------------------------------------

future_df["predicted_sales"] = final_model.predict(
    future_df[FEATURES]
)

# Prevent negative sales predictions
future_df["predicted_sales"] = future_df["predicted_sales"].clip(lower=0)

# ------------------------------------------------------------
# 4. Show result
# ------------------------------------------------------------

total_forecast = future_df["predicted_sales"].sum()

print()
print("========================================")
print("      NEXT-MONTH SALES FORECAST")
print("========================================")
print("Forecast month :", forecast_month)
print("Predicted sales:", total_forecast)
print("========================================")

display(
    future_df[
        ["month", "store_nbr", "family", "predicted_sales"]
    ].head(20)
)

StatementMeta(, 81a1f730-ee43-42bd-ba9c-046c220080b9, 7, Finished, Available, Finished, False)

NameError: name 'pdf' is not defined